# ingest Json 파일 잘 만들어내는지 확인

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트 경로를 Python 경로에 추가
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

import requests
import xml.etree.ElementTree as ET
import json

# 모듈 리로드 (코드 수정 후 다시 테스트할 때 필요)
import importlib
from rag.etl.step01_ingest.pmc_ingest_common import pmc_parsing
importlib.reload(pmc_parsing)
from rag.etl.step01_ingest.pmc_ingest_common.pmc_parsing import extract_article_info

# PMC OAI API 엔드포인트
PMC_OAI_ENDPOINT = "https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi"

# 테스트할 PMCID (테이블이 포함된 문서)
test_pmcid = "PMC12529775" # 테이블, 수식 이미지 확인용
# test_pmcid = "PMC11764125" # 피겨 이미지 확인용

# PMC OAI API로 문서 가져오기
params = {
    "verb": "GetRecord",
    "identifier": f"oai:pubmedcentral.nih.gov:{test_pmcid.replace('PMC', '')}",
    "metadataPrefix": "pmc"
}

print(f"테스트 문서: {test_pmcid}")
print(f"API 요청 중...")

response = requests.get(PMC_OAI_ENDPOINT, params=params, timeout=30)
response.raise_for_status()

print(f"응답 받음 (길이: {len(response.text)} bytes)")

# XML 파싱
root = ET.fromstring(response.content)

# GetRecord 응답에서 record 찾기
ns_oai = {"oai": "http://www.openarchives.org/OAI/2.0/"}
record = root.find(".//oai:record", ns_oai)

if record is None:
    print("❌ record를 찾을 수 없습니다.")
else:
    print("✓ record 찾음")
    
    # extract_article_info 호출
    article_info = extract_article_info(record)
    
    if article_info:
        print(f"\n✓ 문서 파싱 완료")
        print(f"  제목: {article_info.get('title', '')[:100]}...")
        print(f"  테이블 수: {len(article_info.get('table_captions', []))}")
        
        # 테이블 정보 출력
        for i, table in enumerate(article_info.get('table_captions', [])):
            print(f"\n=== 테이블 {i+1} ===")
            print(f"  ID: {table.get('id')}")
            print(f"  Label: {table.get('label')}")
            print(f"  Caption: {table.get('caption', '')[:100]}...")
            print(f"  Page URL: {table.get('page_url')}")
            
            # 테이블 내용 확인
            content = table.get('content')
            if content:
                print(f"\n  ✓ 테이블 내용 파싱됨!")
                
                # Structured (JSON) 확인
                structured = content.get('structured', {})
                print(f"    헤더 행 수: {len(structured.get('headers', []))}")
                print(f"    데이터 행 수: {len(structured.get('rows', []))}")
                
                # 헤더 출력
                if structured.get('headers'):
                    print(f"\n  [JSON - 헤더]")
                    for h_idx, header_row in enumerate(structured['headers']):
                        header_texts = [cell['text'] for cell in header_row]
                        print(f"    행 {h_idx+1}: {header_texts}")
                
                # 데이터 행 일부 출력 (최대 3행)
                if structured.get('rows'):
                    print(f"\n  [JSON - 데이터 행 (최대 3행)]")
                    for r_idx, row in enumerate(structured['rows'][:3]):
                        row_texts = [cell['text'] for cell in row]
                        print(f"    행 {r_idx+1}: {row_texts}")
                
                # Markdown 출력
                markdown = content.get('markdown', '')
                if markdown:
                    print(f"\n  [Markdown]")
                    print("    " + "\n    ".join(markdown.split('\n')[:6]) + "...")
                
                # Text 출력
                text = content.get('text', '')
                if text:
                    print(f"\n  [Text (검색용)]")
                    print(f"    {text[:200]}...")
                
            else:
                print(f"\n  ❌ 테이블 내용이 파싱되지 않음")
        
        # JSON 저장 (확인용)
        output_file = "test_output.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(article_info, f, indent=2, ensure_ascii=False)
        print(f"\n✓ 결과를 '{output_file}'에 저장했습니다.")
    else:
        print("❌ 문서 파싱 실패")

#  2. 피겨 이미지 가져오기

## html 접근성 확인

In [ ]:
from curl_cffi import requests as crequests

def test_fetch_with_impersonate(pmcid):
    url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/"
    print(f"Fetching {url} with impersonate='chrome'...")
    
    try:
        # impersonate="chrome" 옵션이 핵심입니다
        response = crequests.get(url, impersonate="chrome", timeout=30)
        print(f"Status Code: {response.status_code}")
        
        if response.status_code == 200:
            print("성공! HTML 길이:", len(response.text))
            return True
        else:
            print("실패...")
            return False
    except Exception as e:
        print(f"에러 발생: {e}")
        return False

test_fetch_with_impersonate("PMC11764125")

### 피겨 이미지 url 잘 가져오는지 확인

In [ ]:
from curl_cffi import requests as crequests
from bs4 import BeautifulSoup
from pathlib import Path

def test_fetch_details(pmcid, save_html=True):
    url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/"
    print(f"Fetching {url} with impersonate='chrome'...\n")
    
    try:
        response = crequests.get(url, impersonate="chrome", timeout=30)
        
        if response.status_code == 200:
            print("✅ 요청 성공!")
            
            # HTML 파일로 저장
            if save_html:
                html_filename = f"{pmcid}_full.html"
                with open(html_filename, "w", encoding="utf-8") as f:
                    f.write(response.text)
                print(f"💾 전체 HTML을 '{html_filename}' 파일로 저장했습니다. (크기: {len(response.text):,} bytes)\n")
            
            # HTML 파싱
            soup = BeautifulSoup(response.text, "html.parser")
            
            # 1. 문서 제목 확인
            title = soup.find("title")
            print(f"📄 문서 제목: {title.text.strip() if title else '없음'}\n")
            
            # 2. 이미지 태그 찾기
            imgs = soup.find_all("img")
            print(f"🖼️ 발견된 이미지 태그 수: {len(imgs)}")
            
            # 3. Blob 이미지(우리가 찾는 것) 확인
            blob_imgs = []
            for img in imgs:
                src = img.get("src") or img.get("data-src")
                if src and "/blobs/" in src and "cdn.ncbi" in src:
                    blob_imgs.append(src)
            
            print(f"🎯 유효한 Blob 이미지 수: {len(blob_imgs)}")
            
            if blob_imgs:
                print("\n--- Blob 이미지 URL 예시 (최대 5개) ---")
                for i, url in enumerate(blob_imgs[:5]):
                    print(f"{i+1}. {url}")
            else:
                print("\n⚠️ Blob 이미지를 찾지 못했습니다. (이미지가 없는 문서일 수 있음)")
                
                # 디버깅: 일반 이미지 태그 일부 출력
                print("\n--- 일반 이미지 태그 예시 (최대 3개) ---")
                for i, img in enumerate(imgs[:3]):
                    print(f"{i+1}. {img}")

            return True
        else:
            print(f"❌ 실패: Status Code {response.status_code}")
            return False
            
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        return False

test_fetch_details("PMC11764125")

# 3. 수식 이미지 삽입 인식하기

# 3. 수식 이미지 삽입 인식하기

## chandra - api 유료임
## 구글 비전 - 처음 1,000개 단위/월 무료
uv pip install google-cloud-vision 

## 구글 프로3 - aif에서 계정 주면 시도하기

### 구글비전 - 서비스키 발급받아서 주피터파일과 동일한 경로에 두면 됨. 서비스키 원하면 인하에게 말하세요

In [ ]:
import os
from google.cloud import vision

# 서비스 계정 키 경로 설정 (한 번만 실행하면 됨)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "/Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/"
    "rag/etl/step01_ingest/adept-primer-480007-s2-e34691c5af1c.json"
)

def detect_text_uri(uri):
    """이미지 URL에서 텍스트 감지 (OCR)"""
    client = vision.ImageAnnotatorClient()
    image = vision.Image()
    image.source.image_uri = uri

    # TEXT_DETECTION (일반 텍스트) 또는 DOCUMENT_TEXT_DETECTION (문서/손글씨) 사용
    response = client.document_text_detection(image=image)
    texts = response.text_annotations

    if response.error.message:
        raise Exception(f'{response.error.message}')

    if texts:
        print(f'추출된 텍스트:\n"{texts[0].description}"')
    else:
        print("텍스트를 찾지 못했습니다.")

# 예시: 웹상의 이미지 URL
image_url = 'https://cdn.ncbi.nlm.nih.gov/pmc/blobs/039c/12529775/091c40d9be26/ci3c02049_0012.jpg'
detect_text_uri(image_url)

# 테이블 데이터 처리로직 테스트

In [2]:
import sys
from pathlib import Path

# 프로젝트 루트 설정
PROJECT_ROOT = Path("/Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# JSON → CSV 변환 함수 임포트
from rag.etl.step02_normalize.pmc_nomalize_common.pmc_json_to_csv_main import json_to_csv

# 입력 JSON / 출력 디렉토리 경로 설정
input_json = PROJECT_ROOT / "data/raw/pmc/api_extract/pmc_articles_by_category_new.json"
out_dir = PROJECT_ROOT / "data/processed/pubmed/pmc_csv_test"

print(f"INPUT : {input_json}")
print(f"OUTPUT: {out_dir}")

# 변환 실행
json_to_csv(str(input_json), str(out_dir))
print("✅ 변환 완료. 아래 CSV들을 확인하세요:")
print(f"- {out_dir / 'tables.csv'}")
print(f"- {out_dir / 'articles.csv'} 등")


INPUT : /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/raw/pmc/api_extract/pmc_articles_by_category_new.json
OUTPUT: /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/processed/pubmed/pmc_csv_test
[INFO] Loading JSON from /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/raw/pmc/api_extract/pmc_articles_by_category_new.json ...
  -> Writing articles.csv (10 rows)
  -> Writing sections.csv (228 rows)
  -> Writing equations.csv (4 rows)
  -> Writing figures.csv (65 rows)
  -> Writing tables.csv (5 rows)
  -> Writing references.csv (575 rows)
[DONE] Processing complete. Output saved to '/Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/processed/pubmed/pmc_csv_test'
✅ 변환 완료. 아래 CSV들을 확인하세요:
- /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/processed/pubmed/pmc_csv_test/tables.csv
- /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/processed/pubmed/pmc_csv_test/articles.csv 등


# AZURE - llm 사용하기 테스트

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("OPENAI_API_KEY")


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


## 임베딩 테스트

In [3]:
import sys
from pathlib import Path
import logging
import csv
import time
from dotenv import load_dotenv
import os

# 프로젝트 루트 설정
PROJECT_ROOT = Path("/Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM")
sys.path.insert(0, str(PROJECT_ROOT))

# .env 파일 로드
env_path = PROJECT_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
    os.environ["AZURE_OPENAI_ENDPOINT"] = os.getenv("AZURE_OPENAI_ENDPOINT", "")
    print(f"✅ .env 파일 로드됨: {env_path}")
else:
    print("⚠️ .env 파일을 찾을 수 없습니다.")

# 환경변수 확인
api_key = os.getenv("OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

print("=" * 60)
print("테이블 데이터 임베딩 테스트 (Azure OpenAI 지원)")
print("=" * 60)
print(f"OPENAI_API_KEY: {api_key[:10] if api_key else 'None'}...")
print(f"AZURE_OPENAI_ENDPOINT: {azure_endpoint}")
print()

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

# 테이블 임베딩 모듈 임포트
from rag.etl.step05_embed.pmc_embed_common.embed_tables import (
    embed_text,
    _load_existing_table_ids,
    OUTPUT_FIELDNAMES,
    INPUT_REQUIRED_COLS,
)
from rag.etl.step04_chunk.pmc_chunk_common.chunking import (
    DEFAULT_EMBED_MODEL,
    OPENAI_API_KEY,
    AZURE_OPENAI_ENDPOINT,
    RATE_LIMIT_DELAY,
    to_pgvector_literal,
)
from openai import OpenAI
from tqdm import tqdm

# 테스트 폴더 경로 직접 지정 (pmc_csv_test 사용)
input_csv = PROJECT_ROOT / "data" / "processed" / "pubmed" / "pmc_csv_test" / "tables.csv"
output_csv = PROJECT_ROOT / "data" / "embeddings" / "pubmed" / "article_table_emb.csv"

print(f"입력 CSV: {input_csv}")
print(f"출력 CSV: {output_csv}")
print()

if not input_csv.exists():
    print(f"❌ 입력 파일을 찾을 수 없습니다: {input_csv}")
else:
    # Azure OpenAI 또는 표준 OpenAI 클라이언트 초기화
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY가 설정되어 있지 않습니다 (.env 확인).")
    
    if AZURE_OPENAI_ENDPOINT:
        # Azure OpenAI 사용
        client = OpenAI(
            base_url=AZURE_OPENAI_ENDPOINT,
            api_key=OPENAI_API_KEY
        )
        print(f"✅ Azure OpenAI 클라이언트 초기화 완료")
        print(f"   Base URL: {AZURE_OPENAI_ENDPOINT}")
    else:
        # 표준 OpenAI 사용
        client = OpenAI(api_key=OPENAI_API_KEY)
        print(f"✅ 표준 OpenAI 클라이언트 초기화 완료")
    
    # 출력 디렉터리 생성
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    
    # Resume 모드: 기존 임베딩된 table_id 확인
    existing_ids = set()
    file_exists = output_csv.exists()
    if file_exists:
        existing_ids = _load_existing_table_ids(output_csv)
        if existing_ids:
            print(f"📋 Resume 모드: {len(existing_ids)}개 기존 table_id 발견")
    
    # 입력 CSV 행 수 계산
    total_rows = 0
    with input_csv.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        missing = [c for c in INPUT_REQUIRED_COLS if c not in (reader.fieldnames or [])]
        if missing:
            raise ValueError(f"필수 컬럼이 없습니다: {missing}")
        total_rows = sum(1 for _ in reader)
    
    print(f"📊 총 {total_rows}개 테이블 행 처리 시작\n")
    
    # 출력 파일 열기 (append 모드)
    out_f = output_csv.open("a", encoding="utf-8-sig", newline="")
    writer = csv.DictWriter(out_f, fieldnames=OUTPUT_FIELDNAMES)
    if not file_exists:
        writer.writeheader()
    
    new_count = 0
    skip_count = 0
    
    try:
        with input_csv.open("r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            pbar = tqdm(reader, total=total_rows, desc="[EMBED:Tables]", unit="row")
            
            for row in pbar:
                table_id = (row.get("table_id") or "").strip()
                if not table_id:
                    continue
                
                # Resume: 이미 처리된 행 건너뛰기
                if table_id in existing_ids:
                    skip_count += 1
                    continue
                
                caption = (row.get("table_caption") or "").strip()
                if not caption:
                    skip_count += 1
                    continue
                
                # 임베딩 생성
                try:
                    emb = embed_text(client, caption, model=DEFAULT_EMBED_MODEL)
                except Exception as e:
                    print(f"\n❌ table_id={table_id} 임베딩 실패: {e}")
                    continue
                
                emb_literal = to_pgvector_literal(emb)
                
                writer.writerow({
                    "pmcid": row.get("pmcid", ""),
                    "pmid": row.get("pmid", ""),
                    "table_id": table_id,
                    "table_content": row.get("table_content", ""),
                    "table_caption": caption,
                    "table_url": row.get("table_url", ""),
                    "table_caption_emb": emb_literal,
                })
                out_f.flush()
                new_count += 1
                
                # Rate limit 대응
                if RATE_LIMIT_DELAY and RATE_LIMIT_DELAY > 0:
                    time.sleep(RATE_LIMIT_DELAY)
    
    except KeyboardInterrupt:
        print("\n⚠️ 사용자 인터럽트. 현재까지 저장됨.")
    finally:
        out_f.close()
    
    print(f"\n✅ 완료: new={new_count}, skipped={skip_count}")
    print(f"📁 출력 파일: {output_csv}")
    
    # 결과 확인
    import pandas as pd
    if output_csv.exists() and new_count > 0:
        df = pd.read_csv(output_csv)
        print(f"\n📊 결과 요약:")
        print(f"  - 총 테이블 수: {len(df)}개")
        print(f"  - 컬럼: {list(df.columns)}")
        print(f"\n📝 샘플 데이터 (처음 3개):")
        print(df.head(3).to_string())
    else:
        print("\n⚠️ 출력 파일이 생성되지 않았거나 데이터가 없습니다.")
        

✅ .env 파일 로드됨: /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/.env
테이블 데이터 임베딩 테스트 (Azure OpenAI 지원)
OPENAI_API_KEY: EtWqpPGtlm...
AZURE_OPENAI_ENDPOINT: https://helixops-agent.openai.azure.com/openai/v1

입력 CSV: /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/processed/pubmed/pmc_csv_test/tables.csv
출력 CSV: /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/embeddings/pubmed/article_table_emb.csv

✅ Azure OpenAI 클라이언트 초기화 완료
   Base URL: https://helixops-agent.openai.azure.com/openai/v1
📊 총 5개 테이블 행 처리 시작



[EMBED:Tables]: 100%|██████████| 5/5 [00:03<00:00,  1.52row/s]



✅ 완료: new=5, skipped=0
📁 출력 파일: /Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/data/embeddings/pubmed/article_table_emb.csv

📊 결과 요약:
  - 총 테이블 수: 5개
  - 컬럼: ['pmcid', 'pmid', 'table_id', 'table_content', 'table_caption', 'table_url', 'table_caption_emb']

📝 샘플 데이터 (처음 3개):
         pmcid      pmid         table_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

# Gemini (Google Vertex AI) - 팀계정 테스트

In [6]:
from google import genai
from google.genai.types import HttpOptions
client = genai.Client(
    vertexai=True,          # ⭐ 필수
    project="helixops",     # billing 붙은 프로젝트
    location="us-central1",
    http_options=HttpOptions(api_version="v1")
)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How does AI work?"
)
print(response.text)


[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1/projects/helixops/locations/us-central1/publishers/google/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, problem-solving, perception, and understanding language.

At its core, modern AI (especially the most successful type, **Machine Learning**) works by **identifying patterns in data** and then using those patterns to make predictions, classifications, or decisions on new, unseen data.

Here's a breakdown of the key steps and concepts:

### 1. Data, Data, Data!

*   **The Foundation:** AI systems don't "think" out of the box; they learn from experience, much like humans do. This "experience" comes in the form of vast amounts of data.
*   **Examples:**
    *   For an AI that recognizes cats: millions of images labeled "cat" or "not cat."
    *   For a language translation AI: millions of text pairs in two languages.
    *   For a recommendation system: millions of user preferences, purchase histories, and item characte

# Gemini (Google Vertex AI) - 개인계정 테스트 - 새로운 계정으로 다시 구글클라우드 시작해서 300달러 크래딧 받아야 함

In [7]:
from google import genai
from google.genai.types import HttpOptions
client = genai.Client(
    vertexai=True,          # ⭐ 필수
    project="enapeace",     # billing 붙은 프로젝트
    location="us-central1",
    http_options=HttpOptions(api_version="v1")
)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How does AI work?"
)
print(response.text)


[INFO] AFC is enabled with max remote calls: 10.
/Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/.venv/lib/python3.12/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
[INFO] HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1/projects/enapeace/locations/us-central1/publishers/google/models/gemini-2.5-flash:generateContent "HTTP/1.1 403 Forbidden"


ClientError: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Permission denied on resource project enapeace.', 'status': 'PERMISSION_DENIED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'CONSUMER_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'aiplatform.googleapis.com', 'consumer': 'projects/enapeace', 'containerInfo': 'enapeace'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'Permission denied on resource project enapeace.'}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Google developers console', 'url': 'https://console.developers.google.com'}]}]}}